# M* BAGEL: six-request staggered profiling trace

This notebook performs one unmeasured warm-up followed by **six measured 1024×1024 requests staggered by eight seconds** on one A100 80 GB GPU. It records scheduler readiness, component execution, CUDA-event durations, request latency, GPU utilization, and memory. This is an instrumentation/debug run—not statistical performance evidence.

In [ ]:
from google.colab import drive
from datetime import datetime, timezone
from pathlib import Path
import json, os, subprocess, sys

drive.mount('/content/drive')
RUN_ID = 'profile_debug_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
FORK_URL = 'https://github.com/yegurla-sumanth/mstar.git'
GIT_REF = 'bagel-baseline'
MSTAR_BASE = '7fa8b8e72e26097979805eca8fe40930e088ff64'
REPO = Path('/content') / f'mstar_{RUN_ID}'
CACHE_DIR = Path('/content/hf_cache_bagel')
RESULT_DIR = Path('/content/drive/MyDrive/CMU_11868_BAGEL') / RUN_ID
PROFILE_DIR = RESULT_DIR / 'staggered_6'
RESULT_DIR.mkdir(parents=True, exist_ok=False)
print({'run_id': RUN_ID, 'repo': str(REPO), 'results': str(RESULT_DIR)})

In [ ]:
# Resolve and record the exact source before installing or downloading weights.
subprocess.run(['git', 'clone', '--branch', GIT_REF, '--single-branch', FORK_URL, str(REPO)], check=True)
head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
subprocess.run(['git', 'merge-base', '--is-ancestor', MSTAR_BASE, head], cwd=REPO, check=True)
source = {
    'fork_url': FORK_URL,
    'requested_ref': GIT_REF,
    'resolved_commit': head,
    'base_commit': MSTAR_BASE,
    'git_status': subprocess.check_output(['git', 'status', '--porcelain=v1'], cwd=REPO, text=True),
}
(RESULT_DIR / 'source.json').write_text(json.dumps(source, indent=2) + '\n')
subprocess.run([
    sys.executable, str(REPO / 'experiments/bagel/scripts/environment_probe.py'),
    '--output', str(RESULT_DIR / 'preinstall_environment.json'),
    '--disk-path', '/content',
    '--require-gpu-memory-gib', '75',
    '--require-host-memory-gib', '50',
    '--require-disk-free-gib', '70',
], check=True)
print('Preflight passed at commit', head)

In [ ]:
# Install the pinned serving environment plus the plotting dependency.
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'uv==0.10.0'], check=True)
UV = subprocess.check_output([sys.executable, '-c', 'import shutil; print(shutil.which("uv"))'], text=True).strip()
VENV = REPO / '.venv-bagel'
subprocess.run([UV, 'venv', '--python', '3.12', '--seed', str(VENV)], check=True)
VENV_PY = VENV / 'bin/python'
constraints = REPO / 'experiments/bagel/constraints-colab-cu128.txt'
subprocess.run([
    UV, 'pip', 'install', '--python', str(VENV_PY), '--torch-backend=cu128',
    '--constraint', str(constraints), '-e', f'{REPO}[bagel]', 'matplotlib==3.10.7',
], check=True)
subprocess.run([UV, 'pip', 'check', '--python', str(VENV_PY)], check=True)
ninja_path = VENV / 'bin/ninja'
assert ninja_path.exists(), f'Missing required FlashInfer JIT tool: {ninja_path}'
subprocess.run([str(ninja_path), '--version'], check=True)
subprocess.run([str(VENV_PY), str(REPO / 'test/modular/test_bagel_checkpoint_revision.py')], cwd=REPO, check=True)
subprocess.run([str(VENV_PY), str(REPO / 'test/modular/test_execution_trace.py')], cwd=REPO, check=True)
import_check = subprocess.check_output([
    str(VENV_PY), '-c',
    'import os; os.environ["MPLBACKEND"]="Agg"; import torch, flashinfer, matplotlib; assert torch.cuda.is_available(); x=torch.ones((16,16),device="cuda",dtype=torch.bfloat16); y=x@x; torch.cuda.synchronize(); print(torch.__version__, torch.version.cuda, flashinfer.__version__, matplotlib.__version__, float(y[0,0]))'
], text=True)
(RESULT_DIR / 'gpu_import_check.txt').write_text(import_check)
freeze = subprocess.check_output([UV, 'pip', 'freeze', '--python', str(VENV_PY)], text=True)
(RESULT_DIR / 'resolved_requirements.txt').write_text(freeze)
subprocess.run([
    str(VENV_PY), str(REPO / 'experiments/bagel/scripts/environment_probe.py'),
    '--output', str(RESULT_DIR / 'installed_environment.json'), '--disk-path', '/content',
    '--require-gpu-memory-gib', '75', '--require-host-memory-gib', '50',
    '--require-disk-free-gib', '60',
], check=True)
print('Environment installation and import preflight passed')

In [ ]:
# Download the same exact BAGEL checkpoint used by the successful smoke run.
subprocess.run([
    str(VENV_PY), str(REPO / 'experiments/bagel/scripts/download_checkpoint.py'),
    '--cache-dir', str(CACHE_DIR),
    '--manifest', str(RESULT_DIR / 'checkpoint_manifest.json'),
], check=True)
import yaml
manifest = json.loads((RESULT_DIR / 'checkpoint_manifest.json').read_text())
pinned_config = yaml.safe_load((REPO / 'experiments/bagel/configs/bagel_single_gpu_pinned.yaml').read_text())
assert manifest['resolved_revision'] == pinned_config['model_kwargs']['checkpoint_revision']
print('Pinned checkpoint is available at', manifest['resolved_revision'])

In [ ]:
# One warm-up is excluded; six measured requests then arrive eight seconds apart.
subprocess.run([
    str(VENV_PY), str(REPO / 'experiments/bagel/scripts/run_profile.py'),
    '--repo-root', str(REPO),
    '--cache-dir', str(CACHE_DIR),
    '--output-dir', str(PROFILE_DIR),
    '--num-requests', '6',
    '--stagger-seconds', '8',
], check=True)
print('Staggered profiling orchestrator completed')

In [ ]:
# Render and display aligned diagnostic plots.
PLOT_DIR = PROFILE_DIR / 'plots'
subprocess.run([
    str(VENV_PY), str(REPO / 'experiments/bagel/scripts/plot_profile.py'),
    '--input-dir', str(PROFILE_DIR),
    '--output-dir', str(PLOT_DIR),
], check=True)
from IPython.display import Image, display
for filename in (
    'request_component_timeline.png',
    'scheduler_component_timeline.png',
    'gpu_telemetry.png',
    'scheduler_grouping_diagnostics.png',
):
    print(filename)
    display(Image(filename=str(PLOT_DIR / filename)))
profile = json.loads((PROFILE_DIR / 'profile_summary.json').read_text())
run = json.loads((PROFILE_DIR / 'run_summary.json').read_text())
plots = json.loads((PLOT_DIR / 'plot_manifest.json').read_text())
assert profile['status'] == 'success', profile
assert run['status'] == 'success', run
print(json.dumps({
    'resolved_commit': head,
    'server_ready_seconds': run['server_ready_seconds'],
    'measured_latencies_seconds': [round(r['client_elapsed_seconds'], 3) for r in profile['measured_requests']],
    'trace_component_completions': plots['component_completion_count'],
    'plot_status': plots['status'],
    'plot_warnings': plots['warnings'],
    'shutdown': run['shutdown'],
    'result_dir': str(RESULT_DIR),
}, indent=2))

Save or share the four plots and the printed summary for interpretation. Do not treat this six-request trace as a throughput result. After confirming all artifacts exist in Google Drive, use **Runtime → Disconnect and delete runtime**.